This project demonstrates an AI content moderation and safety system using Microsoft Azure AI Content Safety. The system analyzes toxic, harmful, and offensive text samples to identify unsafe content categories including hate speech and violent language. The project also demonstrates how severity scoring and moderation decisions can be integrated into real-world AI safety and automated content filtering workflows using Microsoft Azure AI APIs.

Source dataset : https://www.sciencedirect.com/science/article/pii/S2352340925008741

Microsoft Azure AI API Content Safety Code : https://contentsafety.cognitive.azure.com/text

# EXPLANATION


| Severity Level | Category      | Meaning                                       | Moderation Action |
| -------------- | ------------- | --------------------------------------------- | ----------------- |
| 0              | Safe          | No harmful or unsafe content detected         | Accept            |
| 1              | Very Low Risk | Minimal harmful indication detected           | Accept / Monitor  |
| 2              | Low Risk      | Mild harmful or offensive content detected    | Warning / Monitor |
| 3              | Medium Risk   | Moderate harmful content detected             | Restrict / Review |
| 4              | High Risk     | Dangerous or clearly harmful content detected | Reject / Block    |


 # REAL CASE 1

In [1]:
import pandas as pd
df = pd.read_csv('/content/bilingual_hatespeech_ms_en_v2.csv')
display(df.head())

,text,label,source,lang
0,<number> u0 laughing my ass off wow fuck you,0,HateXplain,en
1,<number> th century mayhem and lawlessness had...,0,HateXplain,en
2,<number> h ada retard au work shessh,0,HateXplain,en
3,<number> stop that wave feminism let us vote a...,0,HateXplain,en
4,<number> okay do not put out patch notes and y...,0,HateXplain,en


In [2]:
for index, row in df.iterrows():
    print(f"Row {index}: {row['text']}")

Streaming output truncated to the last 5000 lines.
Row 21985: oh baiklah ! kopi jelas malu untuk mempamerkan perbezaan , penting lagi saya akan dan saya berdiri dengan setiap perkataan itu . apa tempat yang kacau sekarang ini , apabila menyiarkan kebenaran mendapat satu blok . bercakap
Row 21986: baik dia bandingkan cawangan hsbc kubu dengan rumah rakyat miskin dekat kampung banggol pak suhu alang alang bodoh , baik bodoh maksima terus
Row 21987: kau bodoh mana ada orang pakai komputer lagi , sekarang semua pakai laptop dengan telefon abang teksi , semoga abang teksi itu jadi oranglah aku malas hendak berdebat masa itu sebab tidak ada guna bukannya dia faham pun kalau aku terangkan taca bcasso lqj
Row 21988: <user> bangsat , masuk notif pun enggak
Row 21989: saya berkembang sangat lelah daripada logik yang cacat di balik dasardasar yang semakin fasis ini . wikipedia perlahanlahan menjadi republik pisang . dan kita semua tahu bahawa republik pisang rentan terhadap revolusi dan rampasan 

# REAL CASE 1

text : kuangajo betul , privasi orang taik dia sudah dia baling mercun rumah orang itu tak privasi orang ke bodoh , ini mesti depan mak bapa pijak semut tak mati belakang mengalahkan setan

In [3]:
#
# Copyright (c) Microsoft. All rights reserved.
# To learn more, please visit the documentation - Quickstart: Azure Content Safety: https://aka.ms/acsstudiodoc
#

import enum
import json
import requests
from typing import Union


class MediaType(enum.Enum):
    Text = 1
    Image = 2


class Category(enum.Enum):
    Hate = 1
    SelfHarm = 2
    Sexual = 3
    Violence = 4


class Action(enum.Enum):
    Accept = 1
    Reject = 2


class DetectionError(Exception):
    def __init__(self, code: str, message: str) -> None:
        """
        Exception raised when there is an error in detecting the content.

        Args:
        - code (str): The error code.
        - message (str): The error message.
        """
        self.code = code
        self.message = message

    def __repr__(self) -> str:
        return f"DetectionError(code={self.code}, message={self.message})"


class Decision(object):
    def __init__(
        self, suggested_action: Action, action_by_category: dict[Category, Action]
    ) -> None:
        """
        Represents the decision made by the content moderation system.

        Args:
        - suggested_action (Action): The suggested action to take.
        - action_by_category (dict[Category, Action]): The action to take for each category.
        """
        self.suggested_action = suggested_action
        self.action_by_category = action_by_category


class ContentSafety(object):
    def __init__(self, endpoint: str, subscription_key: str, api_version: str) -> None:
        """
        Creates a new ContentSafety instance.

        Args:
        - endpoint (str): The endpoint URL for the Content Safety API.
        - subscription_key (str): The subscription key for the Content Safety API.
        - api_version (str): The version of the Content Safety API to use.
        """
        self.endpoint = endpoint
        self.subscription_key = subscription_key
        self.api_version = api_version

    def build_url(self, media_type: MediaType) -> str:
        """
        Builds the URL for the Content Safety API based on the media type.

        Args:
        - media_type (MediaType): The type of media to analyze.

        Returns:
        - str: The URL for the Content Safety API.
        """
        if media_type == MediaType.Text:
            return f"{self.endpoint}/contentsafety/text:analyze?api-version={self.api_version}"
        elif media_type == MediaType.Image:
            return f"{self.endpoint}/contentsafety/image:analyze?api-version={self.api_version}"
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def build_headers(self) -> dict[str, str]:
        """
        Builds the headers for the Content Safety API request.

        Returns:
        - dict[str, str]: The headers for the Content Safety API request.
        """
        return {
            "Ocp-Apim-Subscription-Key": self.subscription_key,
            "Content-Type": "application/json",
        }

    def build_request_body(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str],
    ) -> dict:
        """
        Builds the request body for the Content Safety API request.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The request body for the Content Safety API request.
        """
        if media_type == MediaType.Text:
            return {
                "text": content,
                "blocklistNames": blocklists,
            }
        elif media_type == MediaType.Image:
            return {"image": {"content": content}}
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def detect(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str] = [],
    ) -> dict:
        """
        Detects unsafe content using the Content Safety API.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The response from the Content Safety API.
        """
        url = self.build_url(media_type)
        headers = self.build_headers()
        request_body = self.build_request_body(media_type, content, blocklists)
        payload = json.dumps(request_body)

        response = requests.post(url, headers=headers, data=payload)
        print(response.status_code)
        print(response.headers)
        print(response.text)

        res_content = response.json()

        if response.status_code != 200:
            raise DetectionError(
                res_content["error"]["code"], res_content["error"]["message"]
            )

        return res_content

    def get_detect_result_by_category(
        self, category: Category, detect_result: dict
    ) -> Union[int, None]:
        """
        Gets the detection result for the given category from the Content Safety API response.

        Args:
        - category (Category): The category to get the detection result for.
        - detect_result (dict): The Content Safety API response.

        Returns:
        - Union[int, None]: The detection result for the given category, or None if it is not found.
        """
        category_res = detect_result.get("categoriesAnalysis", None)
        for res in category_res:
            if category.name == res.get("category", None):
                return res
        raise ValueError(f"Invalid Category {category}")

    def make_decision(
        self,
        detection_result: dict,
        reject_thresholds: dict[Category, int],
    ) -> Decision:
        """
        Makes a decision based on the Content Safety API response and the specified reject thresholds.
        Users can customize their decision-making method.

        Args:
        - detection_result (dict): The Content Safety API response.
        - reject_thresholds (dict[Category, int]): The reject thresholds for each category.

        Returns:
        - Decision: The decision based on the Content Safety API response and the specified reject thresholds.
        """
        action_result = {}
        final_action = Action.Accept
        for category, threshold in reject_thresholds.items():
            if threshold not in (-1, 0, 2, 4, 6):
                raise ValueError("RejectThreshold can only be in (-1, 0, 2, 4, 6)")

            cate_detect_res = self.get_detect_result_by_category(
                category, detection_result
            )
            if cate_detect_res is None or "severity" not in cate_detect_res:
                raise ValueError(f"Can not find detection result for {category}")

            severity = cate_detect_res["severity"]
            action = (
                Action.Reject
                if threshold != -1 and severity >= threshold
                else Action.Accept
            )
            action_result[category] = action
            if action.value > final_action.value:
                final_action = action

        if (
            "blocklistsMatch" in detection_result
            and detection_result["blocklistsMatch"]
            and len(detection_result["blocklistsMatch"]) > 0
        ):
            final_action = Action.Reject

        print(final_action.name)
        print(action_result)

        return Decision(final_action, action_result)


if __name__ == "__main__":

    # Replace the placeholders with your own values
    endpoint = "https://projectcontent.cognitiveservices.azure.com/"
    subscription_key = "#"
    api_version = "2024-09-01"

    # Initialize the ContentSafety object
    content_safety = ContentSafety(endpoint, subscription_key, api_version)

    # Set the media type and blocklists
    media_type = MediaType.Text
    blocklists = []

    # Set the content to be tested
    content = "kuangajo betul , privasi orang taik dia sudah dia baling mercun rumah orang itu tak privasi orang ke bodoh , ini mesti depan mak bapa pijak semut tak mati belakang mengalahkan setan"

    # Detect content safety
    detection_result = content_safety.detect(media_type, content, blocklists)

    # Set the reject thresholds for each category
    reject_thresholds = {
        Category.Hate: 4,
        Category.SelfHarm: 4,
        Category.Sexual: 4,
        Category.Violence: 4,
    }

    # Make a decision based on the detection result and reject thresholds
    decision_result = content_safety.make_decision(detection_result, reject_thresholds)

200
{'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'apim-request-id': 'a1052d8a-0da0-4a7f-9999-1c6706653a2b', 'csp-billing-usage': 'CognitiveServices.ContentSafety.Text:Analyze=1', 'api-supported-versions': '2023-04-30-preview,2023-05-30-preview,2023-10-01,2023-10-15-preview,2023-10-30-preview,2024-02-15-preview,2024-03-10-preview,2024-03-30-preview,2024-09-01,2024-09-15-preview,2024-09-30-preview,2025-09-15-preview', 'azureml-served-by-cluster': 'hyena-eastus-01', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'East US', 'Date': 'Wed, 06 May 2026 03:15:07 GMT'}
{"blocklistsMatch":[],"categoriesAnalysis":[{"category":"Hate","severity":2},{"category":"SelfHarm","severity":0},{"category":"Sexual","severity":0},{"category":"Violence","severity":4}]}
Reject
{<Category.Hate: 1>: <Action.Accept: 1>, <Category.SelfHarm: 2>: <Action.Accept: 1>, <Category.Sexual: 3>: <Action.Ac

- The moderation analysis conducted using Microsoft Azure AI Content Safety detected harmful content within the input text. Based on the moderation result, the system identified a low-level hate-related risk with a severity score of 2 and a medium-to-high violence risk with a severity score of 4. No harmful content was detected in the Self-Harm and Sexual categories, both of which returned a severity score of 0.

- According to the moderation policy and decision rules, the content was classified as Reject because the Violence category exceeded the acceptable safety threshold. The moderation system automatically marked the Violence category with a reject action, while the other categories remained acceptable.

# REAL CASE 2

text : jngan risau , yang judgmental ini selalunya dia sendiri ada tidak selamat . dbuat bodoh sahaja orang macam ini . tidak bernilai anda time atau mental tenaga

In [4]:
#
# Copyright (c) Microsoft. All rights reserved.
# To learn more, please visit the documentation - Quickstart: Azure Content Safety: https://aka.ms/acsstudiodoc
#

import enum
import json
import requests
from typing import Union


class MediaType(enum.Enum):
    Text = 1
    Image = 2


class Category(enum.Enum):
    Hate = 1
    SelfHarm = 2
    Sexual = 3
    Violence = 4


class Action(enum.Enum):
    Accept = 1
    Reject = 2


class DetectionError(Exception):
    def __init__(self, code: str, message: str) -> None:
        """
        Exception raised when there is an error in detecting the content.

        Args:
        - code (str): The error code.
        - message (str): The error message.
        """
        self.code = code
        self.message = message

    def __repr__(self) -> str:
        return f"DetectionError(code={self.code}, message={self.message})"


class Decision(object):
    def __init__(
        self, suggested_action: Action, action_by_category: dict[Category, Action]
    ) -> None:
        """
        Represents the decision made by the content moderation system.

        Args:
        - suggested_action (Action): The suggested action to take.
        - action_by_category (dict[Category, Action]): The action to take for each category.
        """
        self.suggested_action = suggested_action
        self.action_by_category = action_by_category


class ContentSafety(object):
    def __init__(self, endpoint: str, subscription_key: str, api_version: str) -> None:
        """
        Creates a new ContentSafety instance.

        Args:
        - endpoint (str): The endpoint URL for the Content Safety API.
        - subscription_key (str): The subscription key for the Content Safety API.
        - api_version (str): The version of the Content Safety API to use.
        """
        self.endpoint = endpoint
        self.subscription_key = subscription_key
        self.api_version = api_version

    def build_url(self, media_type: MediaType) -> str:
        """
        Builds the URL for the Content Safety API based on the media type.

        Args:
        - media_type (MediaType): The type of media to analyze.

        Returns:
        - str: The URL for the Content Safety API.
        """
        if media_type == MediaType.Text:
            return f"{self.endpoint}/contentsafety/text:analyze?api-version={self.api_version}"
        elif media_type == MediaType.Image:
            return f"{self.endpoint}/contentsafety/image:analyze?api-version={self.api_version}"
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def build_headers(self) -> dict[str, str]:
        """
        Builds the headers for the Content Safety API request.

        Returns:
        - dict[str, str]: The headers for the Content Safety API request.
        """
        return {
            "Ocp-Apim-Subscription-Key": self.subscription_key,
            "Content-Type": "application/json",
        }

    def build_request_body(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str],
    ) -> dict:
        """
        Builds the request body for the Content Safety API request.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The request body for the Content Safety API request.
        """
        if media_type == MediaType.Text:
            return {
                "text": content,
                "blocklistNames": blocklists,
            }
        elif media_type == MediaType.Image:
            return {"image": {"content": content}}
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def detect(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str] = [],
    ) -> dict:
        """
        Detects unsafe content using the Content Safety API.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The response from the Content Safety API.
        """
        url = self.build_url(media_type)
        headers = self.build_headers()
        request_body = self.build_request_body(media_type, content, blocklists)
        payload = json.dumps(request_body)

        response = requests.post(url, headers=headers, data=payload)
        print(response.status_code)
        print(response.headers)
        print(response.text)

        res_content = response.json()

        if response.status_code != 200:
            raise DetectionError(
                res_content["error"]["code"], res_content["error"]["message"]
            )

        return res_content

    def get_detect_result_by_category(
        self, category: Category, detect_result: dict
    ) -> Union[int, None]:
        """
        Gets the detection result for the given category from the Content Safety API response.

        Args:
        - category (Category): The category to get the detection result for.
        - detect_result (dict): The Content Safety API response.

        Returns:
        - Union[int, None]: The detection result for the given category, or None if it is not found.
        """
        category_res = detect_result.get("categoriesAnalysis", None)
        for res in category_res:
            if category.name == res.get("category", None):
                return res
        raise ValueError(f"Invalid Category {category}")

    def make_decision(
        self,
        detection_result: dict,
        reject_thresholds: dict[Category, int],
    ) -> Decision:
        """
        Makes a decision based on the Content Safety API response and the specified reject thresholds.
        Users can customize their decision-making method.

        Args:
        - detection_result (dict): The Content Safety API response.
        - reject_thresholds (dict[Category, int]): The reject thresholds for each category.

        Returns:
        - Decision: The decision based on the Content Safety API response and the specified reject thresholds.
        """
        action_result = {}
        final_action = Action.Accept
        for category, threshold in reject_thresholds.items():
            if threshold not in (-1, 0, 2, 4, 6):
                raise ValueError("RejectThreshold can only be in (-1, 0, 2, 4, 6)")

            cate_detect_res = self.get_detect_result_by_category(
                category, detection_result
            )
            if cate_detect_res is None or "severity" not in cate_detect_res:
                raise ValueError(f"Can not find detection result for {category}")

            severity = cate_detect_res["severity"]
            action = (
                Action.Reject
                if threshold != -1 and severity >= threshold
                else Action.Accept
            )
            action_result[category] = action
            if action.value > final_action.value:
                final_action = action

        if (
            "blocklistsMatch" in detection_result
            and detection_result["blocklistsMatch"]
            and len(detection_result["blocklistsMatch"]) > 0
        ):
            final_action = Action.Reject

        print(final_action.name)
        print(action_result)

        return Decision(final_action, action_result)


if __name__ == "__main__":

    # Replace the placeholders with your own values
    endpoint = "https://projectcontent.cognitiveservices.azure.com/"
    subscription_key = "#"
    api_version = "2024-09-01"

    # Initialize the ContentSafety object
    content_safety = ContentSafety(endpoint, subscription_key, api_version)

    # Set the media type and blocklists
    media_type = MediaType.Text
    blocklists = []

    # Set the content to be tested
    content = "jngan risau , yang judgmental ini selalunya dia sendiri ada tidak selamat . dbuat bodoh sahaja orang macam ini . tidak bernilai anda time atau mental tenaga"

    # Detect content safety
    detection_result = content_safety.detect(media_type, content, blocklists)

    # Set the reject thresholds for each category
    reject_thresholds = {
        Category.Hate: 4,
        Category.SelfHarm: 4,
        Category.Sexual: 4,
        Category.Violence: 4,
    }

    # Make a decision based on the detection result and reject thresholds
    decision_result = content_safety.make_decision(detection_result, reject_thresholds)

200
{'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'apim-request-id': '7f3d9775-57ee-46a4-a7f1-823b15542997', 'csp-billing-usage': 'CognitiveServices.ContentSafety.Text:Analyze=1', 'api-supported-versions': '2023-04-30-preview,2023-05-30-preview,2023-10-01,2023-10-15-preview,2023-10-30-preview,2024-02-15-preview,2024-03-10-preview,2024-03-30-preview,2024-09-01,2024-09-15-preview,2024-09-30-preview,2025-09-15-preview', 'azureml-served-by-cluster': 'hyena-eastus-01', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'East US', 'Date': 'Wed, 06 May 2026 03:15:08 GMT'}
{"blocklistsMatch":[],"categoriesAnalysis":[{"category":"Hate","severity":2},{"category":"SelfHarm","severity":0},{"category":"Sexual","severity":0},{"category":"Violence","severity":0}]}
Accept
{<Category.Hate: 1>: <Action.Accept: 1>, <Category.SelfHarm: 2>: <Action.Accept: 1>, <Category.Sexual: 3>: <Action.Ac

- The moderation analysis performed using Microsoft Azure AI Content Safety identified a low-level hate-related risk with a severity score of 2, while the Self-Harm, Sexual, and Violence categories were classified as safe with severity scores of 0.

- Based on the moderation policy and automated decision rules, the content was classified as Accept because no category exceeded the defined safety threshold. Although a mild hate-related indication was detected, the severity level remained within the acceptable moderation range and did not require content rejection or restriction.

# REAL CASE 3

text : lapar perhatian bila dia sentap , dia hina mereka mereka yang komen dengan gelaran makcik , pakcik dll terbaru yang kes pengawas sekolah yang rambut dia tak ikut peraturan sekolah murung la kunlun lepas sekolah tarah rambut mak bapa sekarang memang mengada ngada

In [5]:
#
# Copyright (c) Microsoft. All rights reserved.
# To learn more, please visit the documentation - Quickstart: Azure Content Safety: https://aka.ms/acsstudiodoc
#

import enum
import json
import requests
from typing import Union


class MediaType(enum.Enum):
    Text = 1
    Image = 2


class Category(enum.Enum):
    Hate = 1
    SelfHarm = 2
    Sexual = 3
    Violence = 4


class Action(enum.Enum):
    Accept = 1
    Reject = 2


class DetectionError(Exception):
    def __init__(self, code: str, message: str) -> None:
        """
        Exception raised when there is an error in detecting the content.

        Args:
        - code (str): The error code.
        - message (str): The error message.
        """
        self.code = code
        self.message = message

    def __repr__(self) -> str:
        return f"DetectionError(code={self.code}, message={self.message})"


class Decision(object):
    def __init__(
        self, suggested_action: Action, action_by_category: dict[Category, Action]
    ) -> None:
        """
        Represents the decision made by the content moderation system.

        Args:
        - suggested_action (Action): The suggested action to take.
        - action_by_category (dict[Category, Action]): The action to take for each category.
        """
        self.suggested_action = suggested_action
        self.action_by_category = action_by_category


class ContentSafety(object):
    def __init__(self, endpoint: str, subscription_key: str, api_version: str) -> None:
        """
        Creates a new ContentSafety instance.

        Args:
        - endpoint (str): The endpoint URL for the Content Safety API.
        - subscription_key (str): The subscription key for the Content Safety API.
        - api_version (str): The version of the Content Safety API to use.
        """
        self.endpoint = endpoint
        self.subscription_key = subscription_key
        self.api_version = api_version

    def build_url(self, media_type: MediaType) -> str:
        """
        Builds the URL for the Content Safety API based on the media type.

        Args:
        - media_type (MediaType): The type of media to analyze.

        Returns:
        - str: The URL for the Content Safety API.
        """
        if media_type == MediaType.Text:
            return f"{self.endpoint}/contentsafety/text:analyze?api-version={self.api_version}"
        elif media_type == MediaType.Image:
            return f"{self.endpoint}/contentsafety/image:analyze?api-version={self.api_version}"
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def build_headers(self) -> dict[str, str]:
        """
        Builds the headers for the Content Safety API request.

        Returns:
        - dict[str, str]: The headers for the Content Safety API request.
        """
        return {
            "Ocp-Apim-Subscription-Key": self.subscription_key,
            "Content-Type": "application/json",
        }

    def build_request_body(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str],
    ) -> dict:
        """
        Builds the request body for the Content Safety API request.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The request body for the Content Safety API request.
        """
        if media_type == MediaType.Text:
            return {
                "text": content,
                "blocklistNames": blocklists,
            }
        elif media_type == MediaType.Image:
            return {"image": {"content": content}}
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def detect(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str] = [],
    ) -> dict:
        """
        Detects unsafe content using the Content Safety API.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The response from the Content Safety API.
        """
        url = self.build_url(media_type)
        headers = self.build_headers()
        request_body = self.build_request_body(media_type, content, blocklists)
        payload = json.dumps(request_body)

        response = requests.post(url, headers=headers, data=payload)
        print(response.status_code)
        print(response.headers)
        print(response.text)

        res_content = response.json()

        if response.status_code != 200:
            raise DetectionError(
                res_content["error"]["code"], res_content["error"]["message"]
            )

        return res_content

    def get_detect_result_by_category(
        self, category: Category, detect_result: dict
    ) -> Union[int, None]:
        """
        Gets the detection result for the given category from the Content Safety API response.

        Args:
        - category (Category): The category to get the detection result for.
        - detect_result (dict): The Content Safety API response.

        Returns:
        - Union[int, None]: The detection result for the given category, or None if it is not found.
        """
        category_res = detect_result.get("categoriesAnalysis", None)
        for res in category_res:
            if category.name == res.get("category", None):
                return res
        raise ValueError(f"Invalid Category {category}")

    def make_decision(
        self,
        detection_result: dict,
        reject_thresholds: dict[Category, int],
    ) -> Decision:
        """
        Makes a decision based on the Content Safety API response and the specified reject thresholds.
        Users can customize their decision-making method.

        Args:
        - detection_result (dict): The Content Safety API response.
        - reject_thresholds (dict[Category, int]): The reject thresholds for each category.

        Returns:
        - Decision: The decision based on the Content Safety API response and the specified reject thresholds.
        """
        action_result = {}
        final_action = Action.Accept
        for category, threshold in reject_thresholds.items():
            if threshold not in (-1, 0, 2, 4, 6):
                raise ValueError("RejectThreshold can only be in (-1, 0, 2, 4, 6)")

            cate_detect_res = self.get_detect_result_by_category(
                category, detection_result
            )
            if cate_detect_res is None or "severity" not in cate_detect_res:
                raise ValueError(f"Can not find detection result for {category}")

            severity = cate_detect_res["severity"]
            action = (
                Action.Reject
                if threshold != -1 and severity >= threshold
                else Action.Accept
            )
            action_result[category] = action
            if action.value > final_action.value:
                final_action = action

        if (
            "blocklistsMatch" in detection_result
            and detection_result["blocklistsMatch"]
            and len(detection_result["blocklistsMatch"]) > 0
        ):
            final_action = Action.Reject

        print(final_action.name)
        print(action_result)

        return Decision(final_action, action_result)


if __name__ == "__main__":

    # Replace the placeholders with your own values
    endpoint = "https://projectcontent.cognitiveservices.azure.com/"
    subscription_key = "#"
    api_version = "2024-09-01"

    # Initialize the ContentSafety object
    content_safety = ContentSafety(endpoint, subscription_key, api_version)

    # Set the media type and blocklists
    media_type = MediaType.Text
    blocklists = []

    # Set the content to be tested
    content = "lapar perhatian bila dia sentap , dia hina mereka mereka yang komen dengan gelaran makcik , pakcik dll terbaru yang kes pengawas sekolah yang rambut dia tak ikut peraturan sekolah murung la kunlun lepas sekolah tarah rambut mak bapa sekarang memang mengada ngada"

    # Detect content safety
    detection_result = content_safety.detect(media_type, content, blocklists)

    # Set the reject thresholds for each category
    reject_thresholds = {
        Category.Hate: 4,
        Category.SelfHarm: 4,
        Category.Sexual: 4,
        Category.Violence: 4,
    }

    # Make a decision based on the detection result and reject thresholds
    decision_result = content_safety.make_decision(detection_result, reject_thresholds)

200
{'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'apim-request-id': '4022a3e7-2ab3-47da-9ad1-9757e510a6ff', 'csp-billing-usage': 'CognitiveServices.ContentSafety.Text:Analyze=1', 'api-supported-versions': '2023-04-30-preview,2023-05-30-preview,2023-10-01,2023-10-15-preview,2023-10-30-preview,2024-02-15-preview,2024-03-10-preview,2024-03-30-preview,2024-09-01,2024-09-15-preview,2024-09-30-preview,2025-09-15-preview', 'azureml-served-by-cluster': 'hyena-eastus-01', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'East US', 'Date': 'Wed, 06 May 2026 03:15:08 GMT'}
{"blocklistsMatch":[],"categoriesAnalysis":[{"category":"Hate","severity":2},{"category":"SelfHarm","severity":0},{"category":"Sexual","severity":2},{"category":"Violence","severity":0}]}
Accept
{<Category.Hate: 1>: <Action.Accept: 1>, <Category.SelfHarm: 2>: <Action.Accept: 1>, <Category.Sexual: 3>: <Action.Ac

- The moderation analysis performed using Microsoft Azure AI Content Safety identified a low-level hate-related risk with a severity score of 2, while the Self-Harm, Sexual, and Violence categories were classified as safe with severity scores of 0.

- Based on the moderation policy and automated decision rules, the content was classified as Accept because no category exceeded the defined safety threshold. Although a mild hate-related indication was detected, the severity level remained within the acceptable moderation range and did not require content rejection or restriction.

# REAL CASE 4

text : shopeemy kenapa shopeexpress makin lembab ya buatlah ada pilihan untuk kurier semula lembab sangat la servis korang

In [6]:
#
# Copyright (c) Microsoft. All rights reserved.
# To learn more, please visit the documentation - Quickstart: Azure Content Safety: https://aka.ms/acsstudiodoc
#

import enum
import json
import requests
from typing import Union


class MediaType(enum.Enum):
    Text = 1
    Image = 2


class Category(enum.Enum):
    Hate = 1
    SelfHarm = 2
    Sexual = 3
    Violence = 4


class Action(enum.Enum):
    Accept = 1
    Reject = 2


class DetectionError(Exception):
    def __init__(self, code: str, message: str) -> None:
        """
        Exception raised when there is an error in detecting the content.

        Args:
        - code (str): The error code.
        - message (str): The error message.
        """
        self.code = code
        self.message = message

    def __repr__(self) -> str:
        return f"DetectionError(code={self.code}, message={self.message})"


class Decision(object):
    def __init__(
        self, suggested_action: Action, action_by_category: dict[Category, Action]
    ) -> None:
        """
        Represents the decision made by the content moderation system.

        Args:
        - suggested_action (Action): The suggested action to take.
        - action_by_category (dict[Category, Action]): The action to take for each category.
        """
        self.suggested_action = suggested_action
        self.action_by_category = action_by_category


class ContentSafety(object):
    def __init__(self, endpoint: str, subscription_key: str, api_version: str) -> None:
        """
        Creates a new ContentSafety instance.

        Args:
        - endpoint (str): The endpoint URL for the Content Safety API.
        - subscription_key (str): The subscription key for the Content Safety API.
        - api_version (str): The version of the Content Safety API to use.
        """
        self.endpoint = endpoint
        self.subscription_key = subscription_key
        self.api_version = api_version

    def build_url(self, media_type: MediaType) -> str:
        """
        Builds the URL for the Content Safety API based on the media type.

        Args:
        - media_type (MediaType): The type of media to analyze.

        Returns:
        - str: The URL for the Content Safety API.
        """
        if media_type == MediaType.Text:
            return f"{self.endpoint}/contentsafety/text:analyze?api-version={self.api_version}"
        elif media_type == MediaType.Image:
            return f"{self.endpoint}/contentsafety/image:analyze?api-version={self.api_version}"
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def build_headers(self) -> dict[str, str]:
        """
        Builds the headers for the Content Safety API request.

        Returns:
        - dict[str, str]: The headers for the Content Safety API request.
        """
        return {
            "Ocp-Apim-Subscription-Key": self.subscription_key,
            "Content-Type": "application/json",
        }

    def build_request_body(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str],
    ) -> dict:
        """
        Builds the request body for the Content Safety API request.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The request body for the Content Safety API request.
        """
        if media_type == MediaType.Text:
            return {
                "text": content,
                "blocklistNames": blocklists,
            }
        elif media_type == MediaType.Image:
            return {"image": {"content": content}}
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def detect(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str] = [],
    ) -> dict:
        """
        Detects unsafe content using the Content Safety API.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The response from the Content Safety API.
        """
        url = self.build_url(media_type)
        headers = self.build_headers()
        request_body = self.build_request_body(media_type, content, blocklists)
        payload = json.dumps(request_body)

        response = requests.post(url, headers=headers, data=payload)
        print(response.status_code)
        print(response.headers)
        print(response.text)

        res_content = response.json()

        if response.status_code != 200:
            raise DetectionError(
                res_content["error"]["code"], res_content["error"]["message"]
            )

        return res_content

    def get_detect_result_by_category(
        self, category: Category, detect_result: dict
    ) -> Union[int, None]:
        """
        Gets the detection result for the given category from the Content Safety API response.

        Args:
        - category (Category): The category to get the detection result for.
        - detect_result (dict): The Content Safety API response.

        Returns:
        - Union[int, None]: The detection result for the given category, or None if it is not found.
        """
        category_res = detect_result.get("categoriesAnalysis", None)
        for res in category_res:
            if category.name == res.get("category", None):
                return res
        raise ValueError(f"Invalid Category {category}")

    def make_decision(
        self,
        detection_result: dict,
        reject_thresholds: dict[Category, int],
    ) -> Decision:
        """
        Makes a decision based on the Content Safety API response and the specified reject thresholds.
        Users can customize their decision-making method.

        Args:
        - detection_result (dict): The Content Safety API response.
        - reject_thresholds (dict[Category, int]): The reject thresholds for each category.

        Returns:
        - Decision: The decision based on the Content Safety API response and the specified reject thresholds.
        """
        action_result = {}
        final_action = Action.Accept
        for category, threshold in reject_thresholds.items():
            if threshold not in (-1, 0, 2, 4, 6):
                raise ValueError("RejectThreshold can only be in (-1, 0, 2, 4, 6)")

            cate_detect_res = self.get_detect_result_by_category(
                category, detection_result
            )
            if cate_detect_res is None or "severity" not in cate_detect_res:
                raise ValueError(f"Can not find detection result for {category}")

            severity = cate_detect_res["severity"]
            action = (
                Action.Reject
                if threshold != -1 and severity >= threshold
                else Action.Accept
            )
            action_result[category] = action
            if action.value > final_action.value:
                final_action = action

        if (
            "blocklistsMatch" in detection_result
            and detection_result["blocklistsMatch"]
            and len(detection_result["blocklistsMatch"]) > 0
        ):
            final_action = Action.Reject

        print(final_action.name)
        print(action_result)

        return Decision(final_action, action_result)


if __name__ == "__main__":

    # Replace the placeholders with your own values
    endpoint = "https://projectcontent.cognitiveservices.azure.com/"
    subscription_key = "#"
    api_version = "2024-09-01"

    # Initialize the ContentSafety object
    content_safety = ContentSafety(endpoint, subscription_key, api_version)

    # Set the media type and blocklists
    media_type = MediaType.Text
    blocklists = []

    # Set the content to be tested
    content = "shopeemy kenapa shopeexpress makin lembab ya buatlah ada pilihan untuk kurier semula lembab sangat la servis korang"

    # Detect content safety
    detection_result = content_safety.detect(media_type, content, blocklists)

    # Set the reject thresholds for each category
    reject_thresholds = {
        Category.Hate: 4,
        Category.SelfHarm: 4,
        Category.Sexual: 4,
        Category.Violence: 4,
    }

    # Make a decision based on the detection result and reject thresholds
    decision_result = content_safety.make_decision(detection_result, reject_thresholds)

200
{'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'apim-request-id': 'f9e5a4f3-2cc5-41a3-8912-fbc776680c55', 'csp-billing-usage': 'CognitiveServices.ContentSafety.Text:Analyze=1', 'api-supported-versions': '2023-04-30-preview,2023-05-30-preview,2023-10-01,2023-10-15-preview,2023-10-30-preview,2024-02-15-preview,2024-03-10-preview,2024-03-30-preview,2024-09-01,2024-09-15-preview,2024-09-30-preview,2025-09-15-preview', 'azureml-served-by-cluster': 'hyena-eastus-01', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'East US', 'Date': 'Wed, 06 May 2026 03:15:09 GMT'}
{"blocklistsMatch":[],"categoriesAnalysis":[{"category":"Hate","severity":0},{"category":"SelfHarm","severity":0},{"category":"Sexual","severity":0},{"category":"Violence","severity":0}]}
Accept
{<Category.Hate: 1>: <Action.Accept: 1>, <Category.SelfHarm: 2>: <Action.Accept: 1>, <Category.Sexual: 3>: <Action.Ac

- The moderation analysis conducted using Microsoft Azure AI Content Safety classified the submitted content as safe across all moderation categories. The analysis results showed severity scores of 0 for Hate, Self-Harm, Sexual, and Violence categories, indicating that no harmful, offensive, or unsafe content was detected.

- Based on the moderation policy and automated decision rules, the content was classified as Accept because all categories remained within the safe moderation threshold. No restriction, filtering, or rejection action was required for the analyzed text.

# REAL CASE 5

text : bodoh bodoh sudah kene blok pun boleh tahu aku tweet apa kesian la babi miskin ini semak namampos mulut sudah lah macam malang

In [7]:
#
# Copyright (c) Microsoft. All rights reserved.
# To learn more, please visit the documentation - Quickstart: Azure Content Safety: https://aka.ms/acsstudiodoc
#

import enum
import json
import requests
from typing import Union


class MediaType(enum.Enum):
    Text = 1
    Image = 2


class Category(enum.Enum):
    Hate = 1
    SelfHarm = 2
    Sexual = 3
    Violence = 4


class Action(enum.Enum):
    Accept = 1
    Reject = 2


class DetectionError(Exception):
    def __init__(self, code: str, message: str) -> None:
        """
        Exception raised when there is an error in detecting the content.

        Args:
        - code (str): The error code.
        - message (str): The error message.
        """
        self.code = code
        self.message = message

    def __repr__(self) -> str:
        return f"DetectionError(code={self.code}, message={self.message})"


class Decision(object):
    def __init__(
        self, suggested_action: Action, action_by_category: dict[Category, Action]
    ) -> None:
        """
        Represents the decision made by the content moderation system.

        Args:
        - suggested_action (Action): The suggested action to take.
        - action_by_category (dict[Category, Action]): The action to take for each category.
        """
        self.suggested_action = suggested_action
        self.action_by_category = action_by_category


class ContentSafety(object):
    def __init__(self, endpoint: str, subscription_key: str, api_version: str) -> None:
        """
        Creates a new ContentSafety instance.

        Args:
        - endpoint (str): The endpoint URL for the Content Safety API.
        - subscription_key (str): The subscription key for the Content Safety API.
        - api_version (str): The version of the Content Safety API to use.
        """
        self.endpoint = endpoint
        self.subscription_key = subscription_key
        self.api_version = api_version

    def build_url(self, media_type: MediaType) -> str:
        """
        Builds the URL for the Content Safety API based on the media type.

        Args:
        - media_type (MediaType): The type of media to analyze.

        Returns:
        - str: The URL for the Content Safety API.
        """
        if media_type == MediaType.Text:
            return f"{self.endpoint}/contentsafety/text:analyze?api-version={self.api_version}"
        elif media_type == MediaType.Image:
            return f"{self.endpoint}/contentsafety/image:analyze?api-version={self.api_version}"
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def build_headers(self) -> dict[str, str]:
        """
        Builds the headers for the Content Safety API request.

        Returns:
        - dict[str, str]: The headers for the Content Safety API request.
        """
        return {
            "Ocp-Apim-Subscription-Key": self.subscription_key,
            "Content-Type": "application/json",
        }

    def build_request_body(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str],
    ) -> dict:
        """
        Builds the request body for the Content Safety API request.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The request body for the Content Safety API request.
        """
        if media_type == MediaType.Text:
            return {
                "text": content,
                "blocklistNames": blocklists,
            }
        elif media_type == MediaType.Image:
            return {"image": {"content": content}}
        else:
            raise ValueError(f"Invalid Media Type {media_type}")

    def detect(
        self,
        media_type: MediaType,
        content: str,
        blocklists: list[str] = [],
    ) -> dict:
        """
        Detects unsafe content using the Content Safety API.

        Args:
        - media_type (MediaType): The type of media to analyze.
        - content (str): The content to analyze.
        - blocklists (list[str]): The blocklists to use for text analysis.

        Returns:
        - dict: The response from the Content Safety API.
        """
        url = self.build_url(media_type)
        headers = self.build_headers()
        request_body = self.build_request_body(media_type, content, blocklists)
        payload = json.dumps(request_body)

        response = requests.post(url, headers=headers, data=payload)
        print(response.status_code)
        print(response.headers)
        print(response.text)

        res_content = response.json()

        if response.status_code != 200:
            raise DetectionError(
                res_content["error"]["code"], res_content["error"]["message"]
            )

        return res_content

    def get_detect_result_by_category(
        self, category: Category, detect_result: dict
    ) -> Union[int, None]:
        """
        Gets the detection result for the given category from the Content Safety API response.

        Args:
        - category (Category): The category to get the detection result for.
        - detect_result (dict): The Content Safety API response.

        Returns:
        - Union[int, None]: The detection result for the given category, or None if it is not found.
        """
        category_res = detect_result.get("categoriesAnalysis", None)
        for res in category_res:
            if category.name == res.get("category", None):
                return res
        raise ValueError(f"Invalid Category {category}")

    def make_decision(
        self,
        detection_result: dict,
        reject_thresholds: dict[Category, int],
    ) -> Decision:
        """
        Makes a decision based on the Content Safety API response and the specified reject thresholds.
        Users can customize their decision-making method.

        Args:
        - detection_result (dict): The Content Safety API response.
        - reject_thresholds (dict[Category, int]): The reject thresholds for each category.

        Returns:
        - Decision: The decision based on the Content Safety API response and the specified reject thresholds.
        """
        action_result = {}
        final_action = Action.Accept
        for category, threshold in reject_thresholds.items():
            if threshold not in (-1, 0, 2, 4, 6):
                raise ValueError("RejectThreshold can only be in (-1, 0, 2, 4, 6)")

            cate_detect_res = self.get_detect_result_by_category(
                category, detection_result
            )
            if cate_detect_res is None or "severity" not in cate_detect_res:
                raise ValueError(f"Can not find detection result for {category}")

            severity = cate_detect_res["severity"]
            action = (
                Action.Reject
                if threshold != -1 and severity >= threshold
                else Action.Accept
            )
            action_result[category] = action
            if action.value > final_action.value:
                final_action = action

        if (
            "blocklistsMatch" in detection_result
            and detection_result["blocklistsMatch"]
            and len(detection_result["blocklistsMatch"]) > 0
        ):
            final_action = Action.Reject

        print(final_action.name)
        print(action_result)

        return Decision(final_action, action_result)


if __name__ == "__main__":

    # Replace the placeholders with your own values
    endpoint = "https://projectcontent.cognitiveservices.azure.com/"
    subscription_key = "#"
    api_version = "2024-09-01"

    # Initialize the ContentSafety object
    content_safety = ContentSafety(endpoint, subscription_key, api_version)

    # Set the media type and blocklists
    media_type = MediaType.Text
    blocklists = []

    # Set the content to be tested
    content = "bodoh bodoh sudah kene blok pun boleh tahu aku tweet apa kesian la babi miskin ini semak namampos mulut sudah lah macam malang"

    # Detect content safety
    detection_result = content_safety.detect(media_type, content, blocklists)

    # Set the reject thresholds for each category
    reject_thresholds = {
        Category.Hate: 4,
        Category.SelfHarm: 4,
        Category.Sexual: 4,
        Category.Violence: 4,
    }

    # Make a decision based on the detection result and reject thresholds
    decision_result = content_safety.make_decision(detection_result, reject_thresholds)

200
{'Transfer-Encoding': 'chunked', 'Content-Type': 'application/json; charset=utf-8', 'apim-request-id': '44ff277b-ff5d-41df-a6be-9cf3714160c7', 'csp-billing-usage': 'CognitiveServices.ContentSafety.Text:Analyze=1', 'api-supported-versions': '2023-04-30-preview,2023-05-30-preview,2023-10-01,2023-10-15-preview,2023-10-30-preview,2024-02-15-preview,2024-03-10-preview,2024-03-30-preview,2024-09-01,2024-09-15-preview,2024-09-30-preview,2025-09-15-preview', 'azureml-served-by-cluster': 'hyena-eastus-01', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'x-ms-region': 'East US', 'Date': 'Wed, 06 May 2026 03:15:09 GMT'}
{"blocklistsMatch":[],"categoriesAnalysis":[{"category":"Hate","severity":4},{"category":"SelfHarm","severity":0},{"category":"Sexual","severity":0},{"category":"Violence","severity":0}]}
Reject
{<Category.Hate: 1>: <Action.Reject: 2>, <Category.SelfHarm: 2>: <Action.Accept: 1>, <Category.Sexual: 3>: <Action.Ac

- The moderation analysis performed using Microsoft Azure AI Content Safety identified harmful hate-related content with a severity score of 4. The remaining categories, including Self-Harm, Sexual, and Violence, were classified as safe with severity scores of 0.

- Based on the moderation policy and automated decision rules, the content was classified as Reject because the Hate category exceeded the acceptable moderation threshold. The moderation system automatically assigned a reject action to the Hate category to prevent harmful or offensive content from being accepted into the platform.